# No preamble

In [ ]:
# Fix IQ imbalance and take a slice for processing

from commstools import Signal, load_npz
from commstools.backend import dispatch
from commstools.equalization import lms, rde, block_lms
from commstools.sync import (
    compensate_iq_imbalance_lowdin,
    correct_carrier_phase,
    correct_frequency_offset,
    correct_timing,
    estimate_timing,
    recover_carrier_phase_bps,
    recover_carrier_phase_pll,
    estimate_frequency_offset_blockwise,
)

signal = load_npz("/home/lokgar/repos/trmhi304-p2p/waveforms/signal_16qam")

_, xp, _ = dispatch(signal.samples)

# signal.plot_waveform(show=True, num_symbols=71 * 2, title="Frame")
# signal.plot_psd(show=True)
digoff = signal.digital_frequency_offset
signal.shift_frequency(-digoff)
signal.resample(up=1, down=2)
# signal.plot_psd(show=True)

_samples = xp.load(
    "/home/lokgar/repos/trmhi304-p2p/captures/capture_16qam_intradyne_long.npy"
)
samples = xp.array([_samples[0] + 1j * _samples[1], _samples[2] + 1j * _samples[3]])

segment_len = signal.samples.shape[-1]
num_tiles = samples.shape[-1] // segment_len
cut_length = num_tiles * segment_len

samples = samples[:, :cut_length]

samples = compensate_iq_imbalance_lowdin(xp.conj(samples))

num_mult = 3

sig = Signal(
    samples=samples[:, : segment_len * num_mult],
    sampling_rate=signal.sampling_rate,
    symbol_rate=signal.symbol_rate,
    mod_scheme=signal.frame.payload_mod_scheme,
    mod_order=signal.frame.payload_mod_order,
    frame=signal.frame,
    source_bits=xp.tile(signal.frame.payload_bits, (1, num_mult)),
    source_symbols=xp.tile(signal.frame.payload_symbols, (1, num_mult)),
    pulse_shape=signal.pulse_shape,
)

f, Pxx = sig.welch_psd(nperseg=sig.samples.shape[-1])
ofst = -f[Pxx[0].argmax()] - digoff
print(f"OFFSET: {ofst}")
sig.samples = correct_frequency_offset(sig.samples, sig.sampling_rate, -ofst)

sig.plot_psd(show=True, title="Frequency offset corrected PSD")


In [ ]:
sig.resample(sps_out=2)
signal.resample(sps_out=2)

sig.plot_psd(show=True, title="Resampled to 2 SpS")
signal.plot_psd(show=True, title="Resampled to 2 SpS Reference")

coarse, fract = estimate_timing(
    sig.samples,
    signal.samples,
    # pulse_shape=sig.pulse_shape,
    sps=sig.sps,
    debug_plot=True,
    threshold=6,
)
sig.samples = correct_timing(sig.samples, coarse, fract, mode="circular")

sig.matched_filter()
sig.plot_psd(show=True, title="Synced and MFed PSD")


In [ ]:
numeq = 5
num_train_symbols = 2**13
num_taps = 75

block_size = 256
step_size = 1e-1

cpr_type = "bps"

cpr_cycle_slip_correction = True
cpr_cycle_slip_history = 128
cpr_cycle_slip_threshold = xp.pi / 4

cpr_bps_test_phases = 1024
cpr_bps_block_size = 32

cpr_pll_bandwidth = 0.2


for i in range(numeq):
    if i == 0:
        res = block_lms(
            samples=sig.samples[:, : num_train_symbols * 2],
            training_symbols=sig.source_symbols[:, :num_train_symbols],
            modulation=sig.mod_scheme,
            order=sig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=sig.sps,
            plot_smoothing=200,
            debug_plot=True,
            block_size=block_size,
            cpr_type=cpr_type,
            cpr_bps_joint_channels=True,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
        )
    elif i == numeq - 1:
        res = block_lms(
            samples=sig.samples[:, : num_train_symbols * 2],
            training_symbols=sig.source_symbols[:, :num_train_symbols],
            modulation=sig.mod_scheme,
            order=sig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=sig.sps,
            plot_smoothing=200,
            debug_plot=True,
            block_size=block_size,
            cpr_type=cpr_type,
            cpr_bps_joint_channels=True,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
            w_init=res.weights,
        )
    else:
        res = block_lms(
            samples=sig.samples[:, : num_train_symbols * 2],
            training_symbols=sig.source_symbols[:, :num_train_symbols],
            modulation=sig.mod_scheme,
            order=sig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=sig.sps,
            plot_smoothing=200,
            debug_plot=False,
            block_size=block_size,
            cpr_type=cpr_type,
            cpr_bps_joint_channels=True,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
            w_init=res.weights,
        )


res = block_lms(
    samples=sig.samples,
    modulation=sig.mod_scheme,
    order=sig.mod_order,
    num_taps=num_taps,
    step_size=step_size,
    sps=sig.sps,
    plot_smoothing=200,
    debug_plot=True,
    block_size=block_size,
    cpr_type=cpr_type,
    cpr_bps_joint_channels=True,
    cpr_bps_test_phases=cpr_bps_test_phases,
    cpr_bps_block_size=cpr_bps_block_size,
    cpr_cycle_slip_correction=cpr_cycle_slip_correction,
    cpr_cycle_slip_history=cpr_cycle_slip_history,
    cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
    w_init=res.weights,
)

eq = sig.copy()
eq.samples = res.y_hat
eq.sampling_rate = sig.symbol_rate

eq.plot_constellation(show=True, overlay_ideal=True)


# CPR
phases_bps = recover_carrier_phase_bps(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    block_size=cpr_bps_block_size,
    num_test_phases=cpr_bps_test_phases,
    cycle_slip_correction=cpr_cycle_slip_correction,
    cycle_slip_history=cpr_cycle_slip_history,
    cycle_slip_threshold=xp.pi / 4,
    joint_channels=True,
)
phases_pll = recover_carrier_phase_pll(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    loop_filter="butterworth",
    loop_bandwidth_normalized=cpr_pll_bandwidth,
    mu=0.2,
    beta=0.00025,
    cycle_slip_correction=cpr_cycle_slip_correction,
    cycle_slip_history=cpr_cycle_slip_history,
    joint_channels=True,
)

# Recover carrier phase

eqfoecpr = eq.copy()

# eqfoecpr.samples = correct_carrier_phase(eqfoecpr.samples, phases_bps)

eqfoecpr.resolve_symbols()
eqfoecpr.resolve_phase_ambiguity()
eqfoecpr.demap_symbols_hard()

eqfoecpr.plot_constellation(show=True, overlay_ideal=True)


eqfoecpr.evm(num_train_symbols=num_train_symbols, mode="blind")
eqfoecpr.ser(num_train_symbols=num_train_symbols)
eqfoecpr.snr(num_train_symbols=num_train_symbols)


In [ ]:
preeqsig = sig.copy()

numeq = 5
num_train_symbols = 2**13
num_taps = 75

step_size = 8e-3
cpr_type = "bps"
cpr_cycle_slip_correction = True
cpr_cycle_slip_history = 128
cpr_cycle_slip_threshold = xp.pi / 4
cpr_bps_test_phases = 1024
cpr_bps_block_size = 16
cpr_joint_channels = True

cpr_pll_bandwidth = 0.2

for i in range(numeq):
    if i == 0:
        res = lms(
            samples=preeqsig.samples[:, : num_train_symbols * 2],
            training_symbols=preeqsig.source_symbols[:, :num_train_symbols],
            modulation=preeqsig.mod_scheme,
            order=preeqsig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=preeqsig.sps,
            plot_smoothing=200,
            debug_plot=True,
            cpr_type=cpr_type,
            cpr_joint_channels=cpr_joint_channels,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
        )
    elif i == numeq - 1:
        res = lms(
            samples=preeqsig.samples[:, : num_train_symbols * 2],
            training_symbols=preeqsig.source_symbols[:, :num_train_symbols],
            modulation=preeqsig.mod_scheme,
            order=preeqsig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=preeqsig.sps,
            plot_smoothing=200,
            debug_plot=True,
            cpr_type=cpr_type,
            cpr_joint_channels=cpr_joint_channels,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
            w_init=res.weights,
        )
    else:
        res = lms(
            samples=preeqsig.samples[:, : num_train_symbols * 2],
            training_symbols=preeqsig.source_symbols[:, :num_train_symbols],
            modulation=preeqsig.mod_scheme,
            order=preeqsig.mod_order,
            num_taps=num_taps,
            step_size=step_size,
            sps=preeqsig.sps,
            plot_smoothing=200,
            debug_plot=True,
            cpr_type=cpr_type,
            cpr_joint_channels=cpr_joint_channels,
            cpr_bps_test_phases=cpr_bps_test_phases,
            cpr_bps_block_size=cpr_bps_block_size,
            cpr_cycle_slip_correction=cpr_cycle_slip_correction,
            cpr_cycle_slip_history=cpr_cycle_slip_history,
            cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
            w_init=res.weights,
        )

res = lms(
    samples=preeqsig.samples,
    training_symbols=preeqsig.source_symbols[:, :num_train_symbols],
    modulation=preeqsig.mod_scheme,
    order=preeqsig.mod_order,
    num_taps=num_taps,
    step_size=step_size,
    sps=preeqsig.sps,
    plot_smoothing=200,
    debug_plot=True,
    cpr_type=cpr_type,
    cpr_joint_channels=cpr_joint_channels,
    cpr_bps_test_phases=cpr_bps_test_phases,
    cpr_bps_block_size=cpr_bps_block_size,
    cpr_cycle_slip_correction=cpr_cycle_slip_correction,
    cpr_cycle_slip_history=cpr_cycle_slip_history,
    cpr_cycle_slip_threshold=cpr_cycle_slip_threshold,
    w_init=res.weights,
)


In [ ]:
eq = preeqsig.copy()
eq.samples = res.y_hat
eq.sampling_rate = preeqsig.symbol_rate

eq.resolve_symbols()
eq.resolve_phase_ambiguity()
eq.demap_symbols_hard()


eq.evm(mode="blind")
eq.ser(num_train_symbols=num_train_symbols)
eq.snr(num_train_symbols=num_train_symbols)

eq.ser(num_train_symbols=num_train_symbols * 2)
eq.snr(num_train_symbols=num_train_symbols * 2)

eq.plot_constellation(show=True, overlay_ideal=True)


In [ ]:
# CPR
phases_bps = recover_carrier_phase_bps(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    block_size=8,
    num_test_phases=1024,
    cycle_slip_correction=cpr_cycle_slip_correction,
    cycle_slip_history=cpr_cycle_slip_history // 2,
    cycle_slip_threshold=xp.pi / 4,
    joint_channels=True,
)
phases_pll = recover_carrier_phase_pll(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    loop_filter="butterworth",
    loop_bandwidth_normalized=cpr_pll_bandwidth,
    mu=0.2,
    beta=0.00025,
    cycle_slip_correction=cpr_cycle_slip_correction,
    cycle_slip_history=cpr_cycle_slip_history,
    joint_channels=True,
)

# Recover carrier phase

eqfoecpr = eq.copy()

eqfoecpr.samples = correct_carrier_phase(eqfoecpr.samples, phases_bps)

eqfoecpr.resolve_symbols()
eqfoecpr.resolve_phase_ambiguity()
eqfoecpr.demap_symbols_hard()

eqfoecpr.plot_constellation(show=True, overlay_ideal=True)


eqfoecpr.evm(mode="blind")
eqfoecpr.ser(num_train_symbols=num_train_symbols)
eqfoecpr.snr(num_train_symbols=num_train_symbols)

eqfoecpr.ser(num_train_symbols=num_train_symbols * 2)
eqfoecpr.snr(num_train_symbols=num_train_symbols * 2)
